# Introduction to PyTorch and NN 

Class [Notebook](https://colab.research.google.com/drive/19F4uQFN7jIbvbVti5Axs5QNMDS6PBrZb?usp=sharing#scrollTo=sT6XaGEQ-Nwq)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt


## Defining the dataset

In [2]:
train_dataset = torchvision.datasets.MNIST(root='./data', 
                                            train=True, 
                                            download=True, 
                                            transform=transforms.ToTensor())
                                        
test_dataset = torchvision.datasets.MNIST(root='./data', 
                                            train=False, 
                                            download=True, 
                                            transform=transforms.ToTensor())
                                        

## Defining the dataloader

In [3]:
batch_size = 100
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, 
                                           batch_size=batch_size, 
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset, 
                                          batch_size=batch_size, 
                                          shuffle=False)

# Defining the model

In [4]:
# MLP1 for MNIST
# 784 -> 16 -> 16 -> 10

class MLP1(nn.Module):
    def __init__(self):
        super(MLP1, self).__init__()
        self.fc1 = nn.Linear(28*28, 16)      # 784 = 28 * 28 
        self.fc2 = nn.Linear(16, 16)
        self.fc3 = nn.Linear(16, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 28*28) 
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

## Defining the loss function, optimizer and device

In [5]:

device = (
    "cuda" if torch.cuda.is_available() 
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using {device} device")

model = MLP1().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=1e-3)
epochs = 10

Using mps device


## Training and Testing Loop

In [6]:
def training_loop(dataloader, model, loss_fn, optimizer, device):

    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Move the tensor to GPU/MPS
        X = X.to(device)
        y = y.to(device)
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()


        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn, device):

    size = len(dataloader.dataset)
    num_batches = len(dataloader) 
    test_loss, correct = 0, 0

    model.eval()
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)

            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size 

    print(f"Test Error : Accuracy {correct * 100}, Avg Loss {test_loss} ")



# Model Architecture

In [7]:
print(f"Model Structure : {model}\n\n")

for name, param in model.named_parameters():
  print(f"Layer : {name} | Size : {param.size()} | Values : {param[:2]}\n")

Model Structure : MLP1(
  (fc1): Linear(in_features=784, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=16, bias=True)
  (fc3): Linear(in_features=16, out_features=10, bias=True)
  (relu): ReLU()
)


Layer : fc1.weight | Size : torch.Size([16, 784]) | Values : tensor([[-0.0144, -0.0102, -0.0264,  ...,  0.0326, -0.0094,  0.0174],
        [-0.0107, -0.0010,  0.0155,  ..., -0.0285,  0.0147, -0.0070]],
       device='mps:0', grad_fn=<SliceBackward0>)

Layer : fc1.bias | Size : torch.Size([16]) | Values : tensor([-0.0055,  0.0284], device='mps:0', grad_fn=<SliceBackward0>)

Layer : fc2.weight | Size : torch.Size([16, 16]) | Values : tensor([[ 0.2050, -0.1408,  0.0966,  0.1790,  0.1580, -0.1019, -0.2349, -0.0235,
         -0.1036,  0.2445,  0.0861, -0.1219, -0.0992,  0.1327, -0.1982,  0.1957],
        [ 0.0780, -0.0098, -0.0197, -0.0133,  0.1033,  0.2050, -0.0005,  0.1524,
         -0.0767,  0.1543, -0.1662, -0.0394,  0.1579, -0.0461, -0.2267, -0.2119]],
       devi

## Staring the training

In [8]:
for e in range(epochs):
    print(f"Epoch {e+1}\n-------------------------------")
    training_loop(train_loader, model, loss_fn, optimizer, device)
    test_loop(test_loader, model, loss_fn, device)

print("Done Training")

Epoch 1
-------------------------------
loss: 2.295962  [    0/60000]
loss: 0.872326  [10000/60000]
loss: 0.585616  [20000/60000]
loss: 0.257041  [30000/60000]
loss: 0.518787  [40000/60000]
loss: 0.356637  [50000/60000]
Test Error : Accuracy 90.16, Avg Loss 0.3476964608952403 
Epoch 2
-------------------------------
loss: 0.342512  [    0/60000]
loss: 0.376373  [10000/60000]
loss: 0.360739  [20000/60000]
loss: 0.246933  [30000/60000]
loss: 0.371511  [40000/60000]
loss: 0.226807  [50000/60000]
Test Error : Accuracy 91.47, Avg Loss 0.29513328803703187 
Epoch 3
-------------------------------
loss: 0.307656  [    0/60000]
loss: 0.272466  [10000/60000]
loss: 0.277659  [20000/60000]
loss: 0.328262  [30000/60000]
loss: 0.236333  [40000/60000]
loss: 0.338631  [50000/60000]
Test Error : Accuracy 92.25999999999999, Avg Loss 0.2696362048108131 
Epoch 4
-------------------------------
loss: 0.171990  [    0/60000]
loss: 0.366873  [10000/60000]
loss: 0.314039  [20000/60000]
loss: 0.264237  [30000/